# 05c Phased schedule

The phased design in 02 moves the 20 nodes six times after the reconnaissance grid. Every move is a field trip, so
this notebook asks how much imaging is lost by moving them only **two to four** times, which layout makes the
best use of those few moves, and how long each deployment should stay out.

Every configuration starts with the same wide phase 1 grid from 02 (`designs.json`, 5 x 4 nodes over 8 x 6 km).
The BDT high is picked from the phase 1 image, then the nodes are moved K times (K = 2, 3, 4), with each phase 2
deployment placed around that pick. Phase 2 layouts are built from the 02 station bank (0.6 km lattice), so the
03 travel-time tables already cover them.

| Layout | Phase 2 deployments |
|---|---|
| `rolling` | K contiguous 5 x 4 patches at 0.6 km that tile one block around the pick (02's scheme, with fewer patches). Short apertures within each deployment. |
| `interleaved` | The same block and the same sites as `rolling`, but each deployment takes every K-th site, so every deployment spans the whole block at a wider spacing. |
| `zoom` | First move: a 5 x 4 grid at 1.2 km centred on the pick. The BDT high is re-picked from everything recorded so far, then the remaining K - 1 moves are rolling 0.6 km patches around the new pick. |

**Why noise is on.** With noise-free echoes, recording time does not change the image at all (05 only uses
"did the pair record together"). Deployment length only matters through noise, so this notebook uses the noise
model of 06: each pair's correlation noise has std `NSR x SIGNAL x sqrt(DAYS_REF / days)`. Two changes from 06
make time matter properly:

- **Picks only use data recorded by then.** The phase 1 pick (and the zoom re-pick) is made from the data
  recorded up to that day, including the Trillium and Mercury pairs, not from three years of permanent data.
- **One noise record per station pair.** Noise is stacked from a fixed weekly record per pair, so the data
  behind a pick is part of the final image, and schedules are compared on the same noise.

**Earth:** the `dome` BDT at a few conductor positions, with **Vp 5 % higher** inside the BDT zone (the
quartz-cemented case in `docs/03_rock_physics_notes.md`). Under the Born approximation the image scales with the
contrast, so +5 % and -5 % give almost the same |image|. This is really a test of a 5 % contrast.

**Three tests:**

- **A, layout:** every layout x K, at one schedule (phase 1 `P1_DAYS_A`, each move `P2_DAYS_A`), plus the 6-move
  rolling design from 02 as the reference.
- **B, deployment length:** one layout, every K, over a grid of phase 1 and phase 2 lengths.
- **C, whole MT area:** can the array image the BDT everywhere within 4 km of the drill site? Compares "tiled"
  designs (no phase 1: the nodes spread over the whole area in 3 deployments) with the phased designs, for the dome
  at every conductor position.

Costs are given as field trips (install + K moves + retrieval, plus any battery-service visits) and node-days.

Same forward model, imaging and scoring as 05/06 (`mi_common.py`). By default (`FAST`) the image and scatterer
grids are coarser and smaller than 05's, so scores are close to, but not exactly comparable with, 05/06. Every
image is saved and shown as sections along the node axis and as maps of the picked BDT top. A timing cell
estimates the run time before the long runs start.

Inputs: everything 05 reads. Outputs: `data/results_05c_schedule.csv`, images in `data/mi_images_05c/`

In [ ]:
# --- Imports ---
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mi_common import MatrixImaging

In [ ]:
# --- Settings ---
PHASE = "p"
DROP = -0.05                         # perturbation() takes a drop, so -0.05 = Vp 5 % higher inside the BDT zone
SCENARIOS = ["pos00_dome", "pos10_dome"]   # dome at the drill site and 4 km off. add "pos02_dome" (2 km off) etc.
NSR = 1.0                            # noise / signal for DAYS_REF days of shared recording (09 treats 1.0 as realistic)
DAYS_REF = 30
N_REAL = 1                           # noise realisations per scenario. 2-3 for final numbers

GEOMETRIES = ["rolling", "interleaved", "zoom"]
N_MOVES = [2, 3, 4]                  # phase 2 deployments (= node moves after phase 1)
REFERENCE = ("rolling", 6)           # 02's design, for comparison

# Test A: layout, at one schedule
P1_DAYS_A, P2_DAYS_A = 84, 42
# Test B: deployment length, one layout (None = best mean coverage in test A)
LENGTH_GEOMETRY = None
P1_DAYS_B = [42, 84]
P2_DAYS_B = [28, 56]

# Field cost
NODE_BATTERY_DAYS = None             # battery life of a node (with any external pack). A deployment longer than
                                     # this needs a service visit, counted as an extra trip. None = no limit
N_NODES = 20

PATCH = (5, 4)                       # nodes along x across the node axis in one deployment
FINE, COARSE = 0.6, 1.2              # km; multiples of the 02 bank spacing
# Test C: the whole MT target area
MT_R = 4.0                           # km around the drill site where the deep conductor (and BDT high) may be
AREA_SCENARIOS = None                # None = the dome at every conductor position in 04 (15)
TILE_SPACINGS = [1.2, 1.8]           # km between tiled nodes; multiples of the 02 bank spacing (0.6 km)
TILE_DEPLOYMENTS = [3]               # deployments of 20 nodes (moves = deployments - 1); e.g. [2, 3, 4]
TILE_DAYS = 70                       # each tiled deployment. 3 x 70 d = phase 1 (84 d) + 3 x 42 d
AREA_PHASED = [("zoom", 3), ("rolling", 3)]   # phased designs to compare, at the test A schedule

RESULTS = "../data/results_05c_schedule.csv"
IMAGE_DIR = "../data/mi_images_05c"

# Speed: grid spacings in km, turned into steps of the 03 travel-time grid. Cost grows with (image points) and
# (scatterers) x (stations). Scatterers must stay finer than ~1/4 of the shortest wavelength (checked below).
# FAST = None uses 05's settings, so 05c images can be compared with 05/06 directly (slow).
FAST = dict(image_dxy=0.3, image_dz=0.1,     # image grid (km)
            scat_dxy=0.2, scat_dz=0.2,       # Born scatterer grid (km)
            im_r=6.5, im_z=(3.0, 8.0),       # image within +-im_r km of the drill site (MT area + margin), im_z deep
            box=9.0)                         # scatterers within +-box km (tapered over the last 2 km)


def grid_settings(fast):
    if not fast:
        return {}
    g = np.load("../data/tt/grid.npz")
    dx, dz = g["x"][1] - g["x"][0], g["z"][1] - g["z"][0]
    st = lambda v, d: max(1, int(round(v / d)))
    return dict(IM_STEP=(st(fast["image_dxy"], dx), st(fast["image_dxy"], dx), st(fast["image_dz"], dz)),
                SC_STEP=(st(fast["scat_dxy"], dx), st(fast["scat_dxy"], dx), st(fast["scat_dz"], dz)),
                IM_R=fast["im_r"], IM_Z=fast["im_z"], BOX=fast["box"])


mi = MatrixImaging(PHASE, DAYS_REF=DAYS_REF, **grid_settings(FAST))
os.makedirs(IMAGE_DIR, exist_ok=True)
d_sc = max((mi.tx[1] - mi.tx[0]) * mi.SC_STEP[0], (mi.tz[1] - mi.tz[0]) * mi.SC_STEP[2])
lam = np.percentile(mi.v[mi.mx][:, mi.my][:, :, mi.mz[mi.SK.min():mi.SK.max() + 1]], 5) / mi.F_BAND[1]
print(mi)
print(f"scatterer spacing {d_sc:.2f} km, shortest wavelength ~{lam:.2f} km"
      + ("" if d_sc <= lam / 4 + 1e-9 else "  <-- coarser than 1/4 wavelength: grid artefacts likely"))

## Phase 2 layouts

All layouts are built on the 02 station bank: lattice index `i` along the node axis (AZ = 120°) and `j` across
it, 0.6 km apart. The block for `rolling` and `interleaved` is K patches arranged as below (K = 6 is 02's
15 x 8 block):

| K | patches along x across | block (sites) | block size (km) |
|---|---|---|---|
| 1 | 1 x 1 | 5 x 4 | 2.4 x 1.8 |
| 2 | 2 x 1 | 10 x 4 | 5.4 x 1.8 |
| 3 | 3 x 1 | 15 x 4 | 8.4 x 1.8 |
| 4 | 2 x 2 | 10 x 8 | 5.4 x 4.2 |
| 6 | 3 x 2 | 15 x 8 | 8.4 x 4.2 |

In [ ]:
# --- Layouts on the station bank ---
SP, C0, AZR = mi.P2["spacing"], np.array(mi.P2["centre0"], float), mi.AZR
P1_IDS = mi.D["designs"]["phased"][0]["stations"]
BLOCK = {1: (1, 1), 2: (2, 1), 3: (3, 1), 4: (2, 2), 6: (3, 2)}   # patches along x across


def to_axis(xy):
    # Map point -> (along, across) km from the bank centre, kept within P2_SEARCH_R as in 02/05.
    d = np.asarray(xy, float) - C0
    if np.hypot(*d) > mi.P2_SEARCH_R:
        d *= mi.P2_SEARCH_R / np.hypot(*d)
    return np.array([d[0] * np.sin(AZR) + d[1] * np.cos(AZR), -d[0] * np.cos(AZR) + d[1] * np.sin(AZR)])


def lattice(centre, n, step):
    # Bank indices (i along, j across) of an n[0] x n[1] grid at step km, snapped as a whole onto the bank.
    m = int(round(step / SP))
    out = []
    for c, nn in zip(centre, n):
        frac = ((nn - 1) * m / 2) % 1                      # 0, or 0.5 when the grid has no middle site
        base = np.round(c / SP - frac)
        out.append(np.rint(base + frac + (np.arange(nn) - (nn - 1) / 2) * m).astype(int))
    return out


def snake(na, nc):
    return [(a, b) for a in range(na) for b in (range(nc) if a % 2 == 0 else reversed(range(nc)))]


def phase2_sites(geometry, K, centre):
    # Lists of bank (i, j) sites, one list per deployment, for K rolling or interleaved deployments around centre.
    na, nc = BLOCK[K]
    I, J = lattice(centre, (PATCH[0] * na, PATCH[1] * nc), FINE)
    if geometry == "rolling":
        return [[(i, j) for i in I[PATCH[0] * a:PATCH[0] * (a + 1)] for j in J[PATCH[1] * b:PATCH[1] * (b + 1)]]
                for a, b in snake(na, nc)]
    if geometry == "interleaved":
        cls = (lambda a, b: (a + b) % K) if nc == 1 else (lambda a, b: (a % na) * nc + b % nc)
        return [[(I[a], J[b]) for a in range(len(I)) for b in range(len(J)) if cls(a, b) == q] for q in range(K)]
    raise ValueError(geometry)


def zoom_first(centre):
    I, J = lattice(centre, PATCH, COARSE)
    return [(i, j) for i in I for j in J]


def ids(sites):
    # Station ids of bank sites (sites in water or off the bank are dropped, as in 05).
    return [mi.BANK[s] for s in sites if s in mi.BANK]


# every layout gives N_NODES distinct sites per deployment
for g in ("rolling", "interleaved"):
    for K in N_MOVES + [REFERENCE[1]]:
        dep = phase2_sites(g, K, (0.0, 0.0))
        assert all(len(set(d)) == N_NODES for d in dep) and len({s for d in dep for s in d}) == N_NODES * K, (g, K)
assert len(set(zoom_first((0.0, 0.0)))) == N_NODES

In [ ]:
# --- Map of each layout, for a BDT high picked at the bank centre (the zoom re-pick assumed to land there too) ---
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]   # deployment 1, 2, ... in order
ST = mi.stations


def example(geometry, K):
    if geometry == "zoom":
        return [zoom_first((0, 0))] + (phase2_sites("rolling", K - 1, (0, 0)) if K > 1 else [])
    return phase2_sites(geometry, K, (0, 0))


with plt.style.context("default"):
    fig, axs = plt.subplots(len(GEOMETRIES), len(N_MOVES), figsize=(3.3 * len(N_MOVES), 3.3 * len(GEOMETRIES)),
                            sharex=True, sharey=True)
    for r, g in enumerate(GEOMETRIES):
        for c, K in enumerate(N_MOVES):
            ax = axs[r, c]
            ax.plot(ST.x[mi.PERMANENT], ST.y[mi.PERMANENT], "^", color="#8a8984", ms=4, label="Trillium / Mercury")
            ax.plot(ST.x[P1_IDS], ST.y[P1_IDS], "o", mfc="none", mec="k", ms=5, label="Phase 1")
            for q, sites in enumerate(example(g, K)):
                s = ids(sites)
                ax.plot(ST.x[s], ST.y[s], "o", color=COL[q], ms=3.5, mec="white", mew=0.5, label=f"Move {q + 1}")
            ax.set(aspect="equal", xlim=(-8, 8), ylim=(-8, 8), title=f"{g}, {K} moves")
            if r == len(GEOMETRIES) - 1:
                ax.set_xlabel("East (km)")
            if c == 0:
                ax.set_ylabel("North (km)")
    h, l = axs[-1, -1].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), frameon=False, fontsize=8)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()

## Noise, recording time and imaging

The noise on pair (i, j) is the stack of its daily correlations. Each pair has one fixed noise record per
realisation: a unit draw per week (`BLOCK_DAYS`), seeded by (realisation, pair) alone. A pair's noise is the sum of
the weeks it recorded, weighted by the days shared in each week, divided by its total shared days. So:

- a pick and the final image of one run see the same noise on the data they share;
- two schedules that share recording days share the same noise (common random numbers). Differences between
  schedules come from the schedules themselves, not from a lucky draw.

Permanent stations record from day 0. The final image uses the whole Trillium deployment (`DEPLOY_END` days), as
in 05/06.

In [ ]:
# --- Reference signal (as in 06): rms echo of the central dome recorded by the grid design, noise-free ---
NF = len(mi.F)
ref_name = SCENARIOS[0]
_ids = sorted(set(mi.PERMANENT) | set(mi.D["designs"]["grid"][0]["stations"]))
_k, _sig = mi.perturbation(ref_name, DROP)
_R = mi.forward(_ids, _k, _sig)
SIGNAL = float(np.sqrt(np.mean(np.abs(_R[:, mi.pair_mask(_ids, mi.windows(mi.D["designs"]["grid"]))]) ** 2)))
LEVEL = NSR * SIGNAL
del _R
print(f"reference {ref_name}: rms signal {SIGNAL:.2e}, noise level {LEVEL:.2e} for {DAYS_REF} days shared")

In [ ]:
# --- Recording windows, echoes and noise ---
def windows(phases, t0, t1):
    # Recording windows clipped to [t0, t1]: permanent stations throughout, nodes during their phase.
    w = {s: [(t0, t1)] for s in mi.PERMANENT}
    for p in phases:
        a, b = max(p["start"], t0), min(p["end"], t1)
        for s in p["stations"]:
            w.setdefault(s, [])
            if b > a:
                w[s].append((a, b))
    return w


class Echoes:
    # Noise-free reflection matrix for one scenario. Station pairs are computed once and kept, so a pick and the
    # final image, and the configurations that share stations, only pay for pairs not seen before.

    def __init__(self, name):
        self.k, self.sig = mi.perturbation(name, DROP)
        self.pos = {}
        self.R = np.zeros((NF, 64, 64), np.complex64)
        self.known = np.zeros((64, 64), bool)

    def _grow(self, n):
        if n > len(self.known):
            m = max(n, 2 * len(self.known))
            R = np.zeros((NF, m, m), np.complex64)
            known = np.zeros((m, m), bool)
            c = len(self.known)
            R[:, :c, :c], known[:c, :c] = self.R, self.known
            self.R, self.known = R, known

    def get(self, ids):
        for s in ids:
            self.pos.setdefault(s, len(self.pos))
        self._grow(len(self.pos))
        sel = np.array([self.pos[s] for s in ids])
        todo = ~self.known[np.ix_(sel, sel)].all(axis=1)          # stations with at least one pair not computed
        if todo.any():
            k = self.k
            T = np.stack([mi.station_tt(s)[0][k] for s in ids]).astype(np.float64)
            A = np.stack([mi.spreading(s, mi.tx[mi.SI[k]], mi.ty[mi.SJ[k]], mi.tz[mi.SK[k]]) for s in ids])
            G = A * np.exp(-2j * np.pi * mi.F[0] * T)                # frequency by frequency, by phase steps
            step = np.exp(-2j * np.pi * (mi.F[1] - mi.F[0]) * T) if NF > 1 else None
            rows = sel[todo]
            for n in range(NF):
                blk = ((G[todo] * self.sig) @ G.T).astype(np.complex64)
                self.R[n][np.ix_(rows, sel)] = blk
                self.R[n][np.ix_(sel, rows)] = blk.T
                if n < NF - 1:
                    G *= step
            self.known[np.ix_(rows, sel)] = self.known[np.ix_(sel, rows)] = True
        return self.R[:, sel][:, :, sel]


BLOCK_DAYS = 7                                         # noise record resolution (days)
EDGES = np.arange(int(np.ceil(mi.DEPLOY_END / BLOCK_DAYS)) + 1) * BLOCK_DAYS


def shared(w1, w2):
    return [(max(a1, a2), min(b1, b2)) for a1, b1 in w1 for a2, b2 in w2 if min(b1, b2) > max(a1, a2)]


def noisy(R, ids, phases, t_now, real):
    # R plus stacked correlation noise for data recorded up to t_now. Returns (R + noise, shared days).
    win = windows(phases, 0, t_now)
    days = mi.overlap_days(ids, win)
    if not LEVEL:
        return R, days
    lo, hi = EDGES[:-1], EDGES[1:]
    acc = np.zeros(R.shape, np.complex64)
    for p, q in zip(*np.nonzero(np.triu(days > 0, 1))):
        d = sum(np.clip(np.minimum(hi, b) - np.maximum(lo, a), 0, None) for a, b in shared(win[ids[p]], win[ids[q]]))
        nz = np.nonzero(d)[0]
        # the pair's weekly noise record, drawn from (realisation, pair) alone; only the weeks needed (same stream)
        Z = np.random.default_rng([real, *sorted((int(ids[p]), int(ids[q])))]).standard_normal(
            (nz[-1] + 1, 2, NF), dtype=np.float32)
        acc[:, p, q] = ((Z[nz, 0] + 1j * Z[nz, 1]) * np.sqrt(d[nz] / 2)[:, None]).sum(axis=0)
    acc += acc.transpose(0, 2, 1)
    std = LEVEL * np.sqrt(DAYS_REF) / np.maximum(days, 1e-9) * (days > 0)
    return (R + acc * std).astype(np.complex64), days


def image_at(echo, phases, t_now, real):
    # Image from every station that has recorded by t_now, with the noise of the data recorded by then.
    ids = sorted(set(mi.PERMANENT) | {s for p in phases if p["start"] < t_now for s in p["stations"]})
    Rn, days = noisy(echo.get(ids), ids, phases, t_now, real)
    return mi.image(ids, Rn, days > 0), ids, days


def pick(echo, name, phases, t_now, real):
    # BDT high picked from the data recorded by t_now; the bank centre if nothing is imaged. (xy, error km, image)
    img = image_at(echo, phases, t_now, real)[0]
    s = mi.score(img, name)
    ok = np.isfinite(s["crest_x"])
    return (np.array([s["crest_x"], s["crest_y"]]) if ok else C0), s["crest_err_km"], img

In [ ]:
# --- One configuration ---
def trips(lengths):
    # Install + one per move + retrieval, plus service visits for deployments longer than the battery lasts.
    extra = 0 if NODE_BATTERY_DAYS is None else sum(int(np.ceil(L / NODE_BATTERY_DAYS)) - 1 for L in lengths)
    return len(lengths) + 1 + extra


_p1_cache = {}


def area_score(img, name):
    # Scores over the whole MT target area (MT_R around the drill site), not just near the BDT high.
    true, est = mi.true_top(name), mi.pick_top(img)
    area = (np.hypot(mi.XX, mi.YY) <= MT_R) & np.isfinite(true)
    good = area & (np.abs(est - true) <= mi.TOP_TOL)
    return dict(area_coverage=good.sum() / max(area.sum(), 1),
                area_imaged=(area & np.isfinite(est)).sum() / max(area.sum(), 1))


def tiled_name(kind, spacing):
    return f"{'tiled' if kind == 'contiguous' else 'tiled-interleaved'} {spacing:g} km"


TILED = {tiled_name(k, sp): (k, sp) for k in ("contiguous", "interleaved") for sp in TILE_SPACINGS}


def tiled_sites(kind, spacing, n_dep):
    # n_dep deployments of N_NODES bank sites on a spacing-km lattice, filling a disc around the drill site.
    # contiguous: the disc cut into strips across the node axis. interleaved: every deployment spans the disc.
    m = int(round(spacing / SP))
    assert abs(m * SP - spacing) < 1e-6, f"spacing must be a multiple of {SP} km"
    lat = sorted((i, j) for i, j in mi.BANK if i % m == 0 and j % m == 0)
    dist = lambda ij: (np.hypot(*ij), np.arctan2(ij[1], ij[0]))
    if kind == "contiguous":
        disc = sorted(lat, key=dist)[:N_NODES * n_dep]
        disc.sort(key=lambda ij: (ij[0], ij[1]))
        return [disc[q * N_NODES:(q + 1) * N_NODES] for q in range(n_dep)]
    cls = (lambda a, b: (a % 2) * 2 + b % 2) if n_dep == 4 else (lambda a, b: (a + b) % n_dep)
    return [sorted([ij for ij in lat if cls(ij[0] // m, ij[1] // m) == q], key=dist)[:N_NODES] for q in range(n_dep)]


def run_tiled(echo, name, real, geometry, n_moves, dep_days):
    kind, spacing = TILED[geometry]
    phases, t = [], 0
    for sites in tiled_sites(kind, spacing, n_moves + 1):
        phases.append(dict(start=t, end=t + dep_days, stations=ids(sites)))
        t += dep_days
    imgs = dict(final=None)
    imgs["final"], all_ids, days = image_at(echo, phases, mi.DEPLOY_END, real)
    return phases, t, imgs, all_ids, days, {}


def run_config(echo, name, real, geometry, K, p1_days, p2_days):
    if geometry in TILED:                                      # no phase 1: K moves = K + 1 deployments
        phases, t, imgs, all_ids, days, extra = run_tiled(echo, name, real, geometry, K, p2_days)
        return finish(name, real, geometry, K, p1_days, p2_days, phases, t, imgs, all_ids, days, extra)
    p1 = dict(start=0, end=p1_days, stations=P1_IDS)
    key = (name, real, p1_days)
    if key not in _p1_cache:                                   # the phase 1 pick is shared by every layout
        _p1_cache[key] = pick(echo, name, [p1], p1_days, real)
    xy1, err1, img1 = _p1_cache[key]
    phases, t, extra = [p1], p1_days, dict(p1_error_km=err1)
    imgs = dict(phase1=img1)

    if geometry == "zoom":
        phases.append(dict(start=t, end=t + p2_days, stations=ids(zoom_first(to_axis(xy1)))))
        t += p2_days
        xy2, err2, imgs["zoom_pick"] = pick(echo, name, phases, t, real)
        extra["zoom_error_km"] = err2
        rest = phase2_sites("rolling", K - 1, to_axis(xy2)) if K > 1 else []
    else:
        rest = phase2_sites(geometry, K, to_axis(xy1))
    for sites in rest:
        phases.append(dict(start=t, end=t + p2_days, stations=ids(sites)))
        t += p2_days

    imgs["final"], all_ids, days = image_at(echo, phases, mi.DEPLOY_END, real)
    return finish(name, real, geometry, K, p1_days, p2_days, phases, t, imgs, all_ids, days, extra)


def finish(name, real, geometry, K, p1_days, p2_days, phases, t, imgs, all_ids, days, extra):
    # Score the final image, save every image, return the results row.
    lengths = [p["end"] - p["start"] for p in phases]
    row = dict(scenario=name, real=real, geometry=geometry, n_moves=K, p1_days=p1_days, p2_days=p2_days, nsr=NSR,
               campaign_days=t, trips=trips(lengths), node_days=N_NODES * sum(lengths),
               n_stations=len(all_ids), n_pairs=int((days > 0).sum() // 2), **mi.score(imgs["final"], name),
               **area_score(imgs["final"], name), **extra)
    nodes = [(s, q) for q, p in enumerate(phases) for s in p["stations"]]   # deployment 0 = phase 1 (if any)
    np.savez_compressed(image_file(name, real, geometry, K, p1_days, p2_days), x=mi.IX, y=mi.IY, z=mi.IZ,
                        node_ids=[s for s, _ in nodes], node_dep=[q for _, q in nodes], **imgs)
    return row


def image_file(name, real, geometry, K, p1_days, p2_days):
    return f"{IMAGE_DIR}/{name}_r{real}_nsr{NSR:g}_{geometry}_K{K}_{p1_days}+{p2_days}.npz"


KEYS = ["scenario", "real", "geometry", "n_moves", "p1_days", "p2_days", "nsr"]


def saved(f):
    # True if the image file exists and was made on the current image grid (so FAST changes force a rerun).
    if not os.path.exists(f):
        return False
    with np.load(f) as im:
        return all(len(im[k]) == len(v) and np.allclose(im[k], v) for k, v in (("x", mi.IX), ("y", mi.IY), ("z", mi.IZ)))


def run_all(configs, results=RESULTS, scenarios=None):
    # configs: (geometry, K, p1_days, p2_days). Every scenario x realisation; rerunning skips rows already saved.
    scenarios = SCENARIOS if scenarios is None else scenarios
    done = pd.read_csv(results) if os.path.exists(results) else pd.DataFrame(columns=KEYS)
    for col in ("area_coverage", "area_imaged"):
        if col not in done:
            done[col] = np.nan
    have = set(map(tuple, done[KEYS].astype(str).values))
    for name in scenarios:
        echo = None
        for real in range(N_REAL):
            for g, K, a, b in configs:
                f = image_file(name, real, g, K, a, b)
                if tuple(map(str, (name, real, g, K, a, b, NSR))) in have and saved(f):
                    row = ((done.scenario == name) & (done.real.astype(int) == real) & (done.geometry == g)
                           & (done.n_moves.astype(int) == K) & (done.p1_days.astype(int) == a)
                           & (done.p2_days.astype(int) == b) & (done.nsr.astype(float) == NSR))
                    if done.loc[row, "area_coverage"].isna().any():      # run saved before the area score existed
                        for k, v in area_score(np.load(f)["final"], name).items():
                            done.loc[row, k] = v
                        done.to_csv(results, index=False)
                    continue
                done = done[~((done.scenario == name) & (done.real.astype(int) == real) & (done.geometry == g)
                              & (done.n_moves.astype(int) == K) & (done.p1_days.astype(int) == a)
                              & (done.p2_days.astype(int) == b) & (done.nsr.astype(float) == NSR))]
                echo = echo or Echoes(name)
                t0 = time.time()
                row = run_config(echo, name, real, g, K, a, b)
                done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
                done.to_csv(results, index=False)
                print(f"{name} r{real} {g:<11} K={K} {a}+{K}x{b} d: coverage {row['coverage']:.2f} "
                      f"({time.time() - t0:.0f} s)", flush=True)
    m = done.nsr.astype(float) == NSR
    want = {(g, int(K), int(a), int(b)) for g, K, a, b in configs}
    sel = [(g, int(K), int(a), int(b)) in want for g, K, a, b in done[["geometry", "n_moves", "p1_days", "p2_days"]]
           .itertuples(index=False)]
    return done[m & np.array(sel, bool) & done.scenario.isin(scenarios)].astype(
        dict(n_moves=int, p1_days=int, p2_days=int, real=int))

## Test A: layout

Every layout x number of moves at one schedule, plus the 6-move reference.

In [ ]:
CONFIGS_A = [(g, K, P1_DAYS_A, P2_DAYS_A) for g in GEOMETRIES for K in N_MOVES] + \
            [(*REFERENCE, P1_DAYS_A, P2_DAYS_A)]
CONFIGS_B_ALL = [(K, a, b) for K in N_MOVES for a in P1_DAYS_B for b in P2_DAYS_B]

# --- Time one image before committing to the full run ---
_e = Echoes(SCENARIOS[0])
_ids = sorted(set(mi.PERMANENT) | set(P1_IDS) | set(ids(phase2_sites("rolling", 3, (0, 0))[0])))
t0 = time.time(); _R = _e.get(_ids); t_fwd = time.time() - t0
t0 = time.time(); mi.image(_ids, _R, np.ones((len(_ids),) * 2, bool)); t_img = time.time() - t0
del _e, _R
AREA_RUN = mi.scenarios(["dome"]) if AREA_SCENARIOS is None else AREA_SCENARIOS
CONFIGS_C = [(g, n - 1, 0, TILE_DAYS) for g in TILED for n in TILE_DEPLOYMENTS] + \
            [(g, K, P1_DAYS_A, P2_DAYS_A) for g, K in AREA_PHASED]
n_runs = len(SCENARIOS) * N_REAL * (len(CONFIGS_A) + len(CONFIGS_B_ALL)) + len(AREA_RUN) * N_REAL * len(CONFIGS_C)
n_img = n_runs * 1.5                                   # final image, plus phase 1 and zoom picks
print(f"one image {t_img:.1f} s, echoes for {len(_ids)} stations {t_fwd:.1f} s")
print(f"about {n_runs} runs, ~{n_img * t_img / 60 + (len(SCENARIOS) + len(AREA_RUN)) * 4 * t_fwd / 60:.0f} min "
      "(already-saved runs are skipped)")

In [ ]:
resA = run_all(CONFIGS_A)

In [ ]:
# --- Summary per layout and number of moves (over scenarios and noise realisations) ---
AGG = dict(coverage_mean=("coverage", "mean"), coverage_worst=("coverage", "min"),
           area_coverage_mean=("area_coverage", "mean"), area_coverage_worst=("area_coverage", "min"),
           crest_err_km=("crest_err_km", "median"), top_err_km=("top_err_km", "median"),
           p1_error_km=("p1_error_km", "median"), trips=("trips", "first"), campaign_days=("campaign_days", "first"),
           node_days=("node_days", "first"))
sumA = resA.groupby(["geometry", "n_moves"]).agg(**AGG)
sumA.round(2)

In [ ]:
# --- Coverage and BDT-high error against the number of moves ---
ref = sumA.loc[REFERENCE]
GCOL = dict(zip(GEOMETRIES, ["#2a78d6", "#eb6834", "#1baf7a"]))
panels = [("coverage_mean", "Mean coverage of BDT top"), ("coverage_worst", "Worst-case coverage"),
          ("crest_err_km", "BDT high error, median (km)")]

with plt.style.context("default"):
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, (col, lab) in zip(axs, panels):
        for g in GEOMETRIES:
            s = sumA.loc[g].reindex(N_MOVES)[col]
            ax.plot(s.index, s.values, "o-", color=GCOL[g], lw=2, ms=8, mec="white", label=g)
        ax.axhline(ref[col], color="#52514e", ls="--", lw=1, label=f"{REFERENCE[1]}-move reference")
        ax.set(xticks=N_MOVES, xlabel="Phase 2 moves (K)", ylabel=lab)
        if col.startswith("coverage"):
            ax.set_ylim(0, 1.02)
        ax.grid(axis="y", color="#e3e2dd", lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
    axs[0].legend(frameon=False, fontsize=8)
    fig.suptitle(f"Test A: phase 1 {P1_DAYS_A} d, then K moves of {P2_DAYS_A} d each (NSR {NSR})", fontsize=10)
    plt.tight_layout()
    plt.show()

### Images, test A

Sections along the node axis (AZ = 120°) through the scenario's BDT high, and map views of the picked BDT top.
Each image is normalised to its own maximum. Dashed: true BDT top. Dots: top picked from the image (shallowest
strong peak in each column, kept only where the column is above `SNR_MIN` x the image median, as scored).

In [ ]:
# --- Image helpers ---
SHOW_SCENARIO, SHOW_REAL = SCENARIOS[-1], 0       # the off-centre dome is the harder case
CMAP = "magma"


def load(name, real, g, K, a, b):
    return np.load(image_file(name, real, g, K, a, b))


def section(img, name, half=5.0):
    # Image, picked top and true top along the node axis through the scenario centre. Outside the image: NaN.
    c = mi.SC["scenarios"][name]["centre"]
    s = np.linspace(-half, half, int(20 * half) + 1)
    px, py = c[0] + s * np.sin(AZR), c[1] + s * np.cos(AZR)
    inside = (px >= mi.IX[0]) & (px <= mi.IX[-1]) & (py >= mi.IY[0]) & (py <= mi.IY[-1])
    jx = np.clip(np.searchsorted(mi.IX, px), 0, len(mi.IX) - 1)
    jy = np.clip(np.searchsorted(mi.IY, py), 0, len(mi.IY) - 1)
    sec = np.where(inside[:, None], img[jx, jy, :] / img.max(), np.nan)
    picked = np.where(inside, mi.pick_top(img)[jx, jy], np.nan)
    true = np.where(inside, mi.true_top(name)[jx, jy], np.nan)
    return s, sec, picked, true


def draw_section(ax, img, name, title):
    s, sec, picked, true = section(img, name)
    ax.pcolormesh(s, mi.IZ, sec.T, cmap=CMAP, shading="auto", vmin=0, vmax=1)
    ax.plot(s, true, color="#7fd3ff", ls="--", lw=1.2)
    ax.plot(s, picked, ".", color="white", ms=2.5)
    ok = np.isfinite(true) | np.isfinite(sec).any(axis=1)
    ax.set(title=title, ylim=(mi.IZ[-1], mi.IZ[0]), xlim=(s[ok][0], s[ok][-1]))
    ax.title.set_fontsize(9)


def draw_map(ax, f, name, title):
    top = mi.pick_top(f["final"])
    pc = ax.pcolormesh(mi.IX, mi.IY, top.T, cmap="viridis_r", vmin=mi.IZ[0], vmax=mi.IZ[-1], shading="auto")
    ax.contour(mi.IX, mi.IY, mi.true_top(name).T, levels=np.arange(4, 7, 0.5), colors="#52514e", linewidths=0.6)
    st = f["node_ids"][f["node_dep"] > 0]
    ax.plot(ST.x[st], ST.y[st], ".", color="k", ms=1.5)
    ax.plot(*mi.SC["scenarios"][name]["centre"], "+", color="#e34948", ms=10, mew=2)
    ax.set(aspect="equal", xlim=(mi.IX[0], mi.IX[-1]), ylim=(mi.IY[0], mi.IY[-1]), title=title)
    ax.title.set_fontsize(9)
    return pc

In [ ]:
# --- Sections: phase 1 alone and the 6-move reference on top, then every layout x K ---
nm, a, b = SHOW_SCENARIO, P1_DAYS_A, P2_DAYS_A
cov = resA.set_index(["scenario", "real", "geometry", "n_moves"]).coverage

with plt.style.context("default"):
    fig, axs = plt.subplots(len(GEOMETRIES) + 1, len(N_MOVES), figsize=(3.6 * len(N_MOVES), 2.5 * (len(GEOMETRIES) + 1)),
                            sharex=True, sharey=True)
    ref_f = load(nm, SHOW_REAL, *REFERENCE, a, b)
    draw_section(axs[0, 0], ref_f["phase1"], nm, f"phase 1 alone ({a} d)")
    draw_section(axs[0, 1], ref_f["final"], nm,
                 f"{REFERENCE[1]}-move reference, cov {cov[nm, SHOW_REAL, REFERENCE[0], REFERENCE[1]]:.2f}")
    axs[0, 2].axis("off")
    for r, g in enumerate(GEOMETRIES):
        for c, K in enumerate(N_MOVES):
            draw_section(axs[r + 1, c], load(nm, SHOW_REAL, g, K, a, b)["final"], nm,
                         f"{g}, {K} moves, cov {cov[nm, SHOW_REAL, g, K]:.2f}")
    for ax in axs[:, 0]:
        ax.set_ylabel("Depth (km)")
    for ax in axs[-1]:
        ax.set_xlabel("Along node axis (km)")
    fig.suptitle(f"{nm}: phase 1 {a} d, then K moves of {b} d", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Map view: depth of the picked BDT top (blank = not imaged). Grey: true top contours. Black: phase 2 nodes ---
with plt.style.context("default"):
    fig, axs = plt.subplots(len(GEOMETRIES), len(N_MOVES), figsize=(3.3 * len(N_MOVES) + 0.8, 3.2 * len(GEOMETRIES)),
                            sharex=True, sharey=True, layout="constrained")
    for r, g in enumerate(GEOMETRIES):
        for c, K in enumerate(N_MOVES):
            pc = draw_map(axs[r, c], load(nm, SHOW_REAL, g, K, a, b), nm, f"{g}, {K} moves")
    for ax in axs[-1]:
        ax.set_xlabel("East (km)")
    for ax in axs[:, 0]:
        ax.set_ylabel("North (km)")
    fig.colorbar(pc, ax=axs, label="Picked BDT top (km)", shrink=0.6).ax.invert_yaxis()
    fig.suptitle(f"{nm}: picked top, + = true BDT high", fontsize=10)
    plt.show()

### Why is a column blank?

A blank column in the maps above can mean three different things. For one configuration, the figure below shows:

1. **Why each column is blank:** below the detection floor (`SNR_MIN` x the image median), or bright but with no
   peak inside the depth window (its maximum is at the top or bottom edge, where the pick does not look).
2. **Column peak / detection floor** (log scale; the black line is the floor). Values just under 1 are close to
   being imaged; values far under 1 are not illuminated.
3. **Depth of each column's brightest point.** Pinned at the top or bottom of the window means the window cuts it.
4. **Illumination:** station pairs that recorded together with their midpoint within 1 km of the column, a rough
   proxy for how many reflection points a flat reflector gets there.

How to fix each one: few pairs means move nodes there (or a wider phase 1); peaks at the window edge mean widening
`FAST["im_z"]`; blanks only along the map edge mean raising `FAST["im_r"]`.

In [ ]:
# --- Diagnose blank columns for one configuration ---
DIAG = ("zoom", 3)                                  # layout, K; uses the test A schedule and SHOW_SCENARIO
f = load(nm, SHOW_REAL, *DIAG, P1_DAYS_A, P2_DAYS_A)
img = f["final"]
peak, floor = img.max(axis=2), mi.SNR_MIN * np.median(img)
a = img / np.maximum(peak[..., None], 1e-30)
interior = np.zeros_like(a, bool)
interior[..., 1:-1] = (a[..., 1:-1] >= a[..., :-2]) & (a[..., 1:-1] >= a[..., 2:]) & (a[..., 1:-1] >= 0.5)
reason = np.where(peak < floor, 1, np.where(~interior.any(axis=2), 2, 0))   # 0 imaged, 1 below floor, 2 edge peak
zpeak = mi.IZ[img.argmax(axis=2)]

# station pairs that recorded together: permanent with anything, nodes with nodes of the same deployment
sx = np.r_[ST.x[mi.PERMANENT].to_numpy(), ST.x[f["node_ids"]].to_numpy()]
sy = np.r_[ST.y[mi.PERMANENT].to_numpy(), ST.y[f["node_ids"]].to_numpy()]
dep = np.r_[np.full(len(mi.PERMANENT), -1), f["node_dep"]]
i, j = np.triu_indices(len(sx), 1)
keep = (dep[i] < 0) | (dep[j] < 0) | (dep[i] == dep[j])
mx, my = (sx[i] + sx[j])[keep] / 2, (sy[i] + sy[j])[keep] / 2
npairs = np.array([[np.sum((mx - x) ** 2 + (my - y) ** 2 <= 1.0) for y in mi.IY] for x in mi.IX])

from matplotlib.colors import ListedColormap, LogNorm
with plt.style.context("default"):
    fig, axs = plt.subplots(1, 4, figsize=(17, 4.4), sharex=True, sharey=True, layout="constrained")
    pc = axs[0].pcolormesh(mi.IX, mi.IY, reason.T, shading="auto", vmin=-0.5, vmax=2.5,
                           cmap=ListedColormap(["#2a78d6", "#d9d8d3", "#eb6834"]))
    cb = fig.colorbar(pc, ax=axs[0], ticks=[0, 1, 2], shrink=0.8)
    cb.ax.set_yticklabels(["imaged", "below floor", "peak at\nwindow edge"])
    pc = axs[1].pcolormesh(mi.IX, mi.IY, (peak / floor).T, shading="auto", cmap="viridis",
                           norm=LogNorm(vmin=0.1, vmax=max(10, (peak / floor).max())))
    axs[1].contour(mi.IX, mi.IY, (peak / floor).T, levels=[1], colors="k", linewidths=1)
    fig.colorbar(pc, ax=axs[1], label="column peak / floor", shrink=0.8)
    pc = axs[2].pcolormesh(mi.IX, mi.IY, zpeak.T, shading="auto", cmap="viridis_r", vmin=mi.IZ[0], vmax=mi.IZ[-1])
    fig.colorbar(pc, ax=axs[2], label="depth of brightest point (km)", shrink=0.8).ax.invert_yaxis()
    pc = axs[3].pcolormesh(mi.IX, mi.IY, npairs.T, shading="auto", cmap="Blues")
    fig.colorbar(pc, ax=axs[3], label="pairs with midpoint within 1 km", shrink=0.8)
    titles = ["Why blank", "Peak vs detection floor", "Depth of column maximum", "Illumination"]
    for ax, t in zip(axs, titles):
        ax.contour(mi.IX, mi.IY, mi.true_top(nm).T, levels=np.arange(4, 7, 0.5), colors="#52514e", linewidths=0.6)
        ax.plot(ST.x[mi.PERMANENT], ST.y[mi.PERMANENT], "^", color="#e34948", ms=4)
        ax.plot(sx[len(mi.PERMANENT):], sy[len(mi.PERMANENT):], ".", color="k", ms=2)
        ax.plot(*mi.SC["scenarios"][nm]["centre"], "+", color="#e34948", ms=10, mew=2)
        ax.set(aspect="equal", xlim=(mi.IX[0], mi.IX[-1]), ylim=(mi.IY[0], mi.IY[-1]), title=t, xlabel="East (km)")
    axs[0].set_ylabel("North (km)")
    fig.suptitle(f"{nm}, {DIAG[0]} {DIAG[1]} moves: triangles = Trillium/Mercury, dots = all nodes (phase 1 + 2)",
                 fontsize=10)
    plt.show()

t = mi.true_top(nm)
c = mi.SC["scenarios"][nm]["centre"]
tgt = (np.hypot(mi.XX - c[0], mi.YY - c[1]) <= mi.TARGET_R) & np.isfinite(t)
print(f"within {mi.TARGET_R:g} km of the BDT high: imaged {np.mean(reason[tgt] == 0):.0%}, "
      f"below floor {np.mean(reason[tgt] == 1):.0%}, peak at window edge {np.mean(reason[tgt] == 2):.0%}")
print(f"target area inside the image box: {tgt.sum()} columns of "
      f"{int(np.pi * mi.TARGET_R ** 2 / ((mi.IX[1] - mi.IX[0]) * (mi.IY[1] - mi.IY[0])))} expected")

## Test B: deployment length

One layout (the best in test A unless `LENGTH_GEOMETRY` is set), every K, over the grid of phase 1 and phase 2
lengths. The phase 1 length sets how well the BDT high is found before the nodes move; the phase 2 length sets
the noise of the dense data.

In [ ]:
GEOM_B = LENGTH_GEOMETRY or sumA.drop(REFERENCE).coverage_mean.groupby("geometry").mean().idxmax()
print("test B layout:", GEOM_B)
CONFIGS_B = [(GEOM_B, K, a, b) for K in N_MOVES for a in P1_DAYS_B for b in P2_DAYS_B]
resB = run_all(CONFIGS_B)
sumB = resB.groupby(["n_moves", "p1_days", "p2_days"]).agg(**AGG)
sumB.round(2)

In [ ]:
# --- Mean coverage over phase 1 x phase 2 length, one panel per K ---
with plt.style.context("default"):
    fig, axs = plt.subplots(1, len(N_MOVES), figsize=(3.6 * len(N_MOVES) + 1, 3.2), sharey=True)
    for ax, K in zip(np.atleast_1d(axs), N_MOVES):
        t = sumB.loc[K].coverage_mean.unstack("p2_days").reindex(index=P1_DAYS_B, columns=P2_DAYS_B)
        im = ax.imshow(t.values, cmap="Blues", vmin=0, vmax=1, origin="lower", aspect="auto")
        for i in range(t.shape[0]):
            for j in range(t.shape[1]):
                v = t.values[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                        color="white" if v > 0.5 else "#0b0b0b")
        ax.set(xticks=range(len(P2_DAYS_B)), xticklabels=P2_DAYS_B, yticks=range(len(P1_DAYS_B)),
               yticklabels=P1_DAYS_B, xlabel="Each phase 2 deployment (days)", title=f"{GEOM_B}, {K} moves")
    np.atleast_1d(axs)[0].set_ylabel("Phase 1 (days)")
    fig.colorbar(im, ax=axs, label="Mean coverage of BDT top", shrink=0.9)
    plt.show()

In [ ]:
# --- Sections for every schedule in test B, one row per number of moves ---
sched = [(a, b) for a in P1_DAYS_B for b in P2_DAYS_B]
covB = resB.set_index(["scenario", "real", "n_moves", "p1_days", "p2_days"]).coverage

with plt.style.context("default"):
    fig, axs = plt.subplots(len(N_MOVES), len(sched), figsize=(3.3 * len(sched), 2.5 * len(N_MOVES)),
                            sharex=True, sharey=True, squeeze=False)
    for r, K in enumerate(N_MOVES):
        for c, (a, b) in enumerate(sched):
            draw_section(axs[r, c], load(nm, SHOW_REAL, GEOM_B, K, a, b)["final"], nm,
                         f"K={K}, {a} + {K} x {b} d, cov {covB[nm, SHOW_REAL, K, a, b]:.2f}")
    for ax in axs[:, 0]:
        ax.set_ylabel("Depth (km)")
    for ax in axs[-1]:
        ax.set_xlabel("Along node axis (km)")
    fig.suptitle(f"{nm}, {GEOM_B}: phase 1 + K moves", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- What each schedule buys: coverage against campaign length, by number of moves ---
MK = {2: "o", 3: "s", 4: "D"}
KCOL = dict(zip(N_MOVES, ["#2a78d6", "#eb6834", "#1baf7a"]))
s = sumB.reset_index()

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(7, 4))
    for K in N_MOVES:
        g = s[s.n_moves == K]
        for a, fill in zip(P1_DAYS_B, ["white", None]):
            h = g[g.p1_days == a]
            ax.plot(h.campaign_days, h.coverage_mean, MK.get(K, "o"), ms=9, mec=KCOL[K], mew=2,
                    mfc=fill or KCOL[K], label=f"{K} moves ({K + 2} trips), phase 1 {a} d")
    ax.axhline(ref.coverage_mean, color="#52514e", ls="--", lw=1,
               label=f"{REFERENCE[1]}-move reference ({int(ref.trips)} trips, {int(ref.campaign_days)} d)")
    ax.set(xlabel="Node campaign length (days)", ylabel="Mean coverage of BDT top", ylim=(0, 1.02))
    ax.grid(color="#e3e2dd", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
# --- All schedules tested, cheapest first: field trips, campaign length, node-days and scores ---
allres = pd.concat([resA, resB]).drop_duplicates(KEYS)
table = (allres.groupby(["geometry", "n_moves", "p1_days", "p2_days"]).agg(**AGG)
         .reset_index().sort_values(["trips", "campaign_days", "coverage_mean"], ascending=[True, True, False]))
print(table.round(2).to_string(index=False))

## Test C: covering the whole MT target area

Tests A and B ask how well a design *finds and images the BDT high*. Here the question is different: **can the
array image the BDT top everywhere within `MT_R` (4 km) of the drill site**, wherever the conductor turns out to
be? The score is `area_coverage`: the fraction of image columns inside that circle whose picked top is within
`TOP_TOL` (0.3 km) of the truth.

Why this needs a different layout: a point on the BDT is imaged by station pairs on *both* sides of it, and a dome
flank reflects energy outwards, so the stations have to reach beyond the area to be imaged by ~2-3 km. Dense
patches on the BDT high (tests A/B) leave the rest of the area, and the far flanks, without that.

**Tiled designs** spread the nodes over a disc around the drill site instead, with no phase 1 and no pick:

| Design | Deployments |
|---|---|
| `tiled S km` | the disc of 20 x n sites at S km spacing, cut into n strips across the node axis; each deployment is one strip |
| `tiled-interleaved S km` | the same disc, but each deployment takes every n-th site, so every deployment spans the whole disc at a wider spacing |

They are compared with the phased designs in `AREA_PHASED` at the test A schedule, over the dome at every conductor
position in 04 (the centre, a 2 km ring and a 4 km ring). Default: 3 tiled deployments of 70 d, which is the same
campaign length as phase 1 + 3 moves of 42 d, with one field trip fewer.

In [ ]:
# --- Tiled layouts. Dashed circle: the MT target area ---
th = np.linspace(0, 2 * np.pi, 200)
with plt.style.context("default"):
    lay = [(g, n) for g in TILED for n in TILE_DEPLOYMENTS]
    fig, axs = plt.subplots(1, len(lay), figsize=(3.4 * len(lay), 3.7), sharex=True, sharey=True, squeeze=False)
    for ax, (g, n) in zip(axs[0], lay):
        ax.plot(ST.x[mi.PERMANENT], ST.y[mi.PERMANENT], "^", color="#8a8984", ms=4, label="Trillium / Mercury")
        for q, sites in enumerate(tiled_sites(*TILED[g], n)):
            s_ = ids(sites)
            ax.plot(ST.x[s_], ST.y[s_], "o", color=COL[q], ms=3.5, mec="white", mew=0.5, label=f"Deployment {q + 1}")
        ax.plot(MT_R * np.cos(th), MT_R * np.sin(th), "--", color="#52514e", lw=1)
        ax.set(aspect="equal", xlim=(-8, 8), ylim=(-8, 8), title=f"{g}, {n} deployments", xlabel="East (km)")
        ax.title.set_fontsize(9)
    axs[0, 0].set_ylabel("North (km)")
    h, l = axs[0, -1].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), frameon=False, fontsize=8)
    plt.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()

In [ ]:
# --- Run every design over every conductor position ---
resC = run_all(CONFIGS_C, scenarios=AREA_RUN)


def label(g, K, a, b):
    if g in TILED:
        return f"{g}, {K + 1} x {b} d"
    return f"{g}, phase 1 {a} d + {K} x {b} d"


resC["design"] = [label(*r) for r in resC[["geometry", "n_moves", "p1_days", "p2_days"]].itertuples(index=False)]
sumC = (resC.groupby("design").agg(**AGG, area_imaged=("area_imaged", "mean"))
        [["trips", "campaign_days", "node_days", "area_coverage_mean", "area_coverage_worst", "area_imaged",
          "coverage_mean", "crest_err_km"]].sort_values("area_coverage_mean", ascending=False))
sumC.round(2)

In [ ]:
# --- Whole-area coverage for each design: one dot per conductor position, bar = mean ---
order = sumC.index[::-1]
ring = {n: int(round(np.hypot(*mi.SC["scenarios"][n]["centre"]))) for n in AREA_RUN}   # 0, 2 or 4 km off
RCOL = {0: "#2a78d6", 2: "#eb6834", 4: "#1baf7a"}

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(8, 0.55 * len(order) + 1.2))
    for y, d in enumerate(order):
        g = resC[resC.design == d]
        ax.plot([g.area_coverage.mean()] * 2, [y - 0.3, y + 0.3], color="#0b0b0b", lw=2)
        for r_ in (0, 2, 4):
            v = g[g.scenario.map(ring) == r_].area_coverage
            ax.plot(v, np.full(len(v), y) + (r_ - 2) * 0.08, "o", color=RCOL[r_], ms=6, mec="white", mew=0.5,
                    label=f"conductor {r_} km from drill site" if y == 0 else None)
    ax.set_yticks(range(len(order)), [f"{d}   ({int(sumC.trips[d])} trips)" for d in order], fontsize=8)
    ax.set(xlim=(0, 1), xlabel=f"Coverage of the BDT top within {MT_R:g} km of the drill site")
    ax.grid(axis="x", color="#e3e2dd", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1), ncol=3)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Where is each design reliable? Fraction of conductor positions for which each column is imaged correctly ---
designs_C = list(sumC.index)
key_of = {label(*c): c for c in CONFIGS_C}

with plt.style.context("default"):
    nc = min(3, len(designs_C))
    nr = int(np.ceil(len(designs_C) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.6 * nc + 0.8, 3.5 * nr), sharex=True, sharey=True,
                            squeeze=False, layout="constrained")
    for ax, d in zip(axs.ravel(), designs_C):
        g, K, a, b = key_of[d]
        hits = []
        for n in AREA_RUN:
            for r_ in range(N_REAL):
                f = load(n, r_, g, K, a, b)
                hits.append(np.abs(mi.pick_top(f["final"]) - mi.true_top(n)) <= mi.TOP_TOL)
        pc = ax.pcolormesh(mi.IX, mi.IY, np.mean(hits, axis=0).T, cmap="Blues", vmin=0, vmax=1, shading="auto")
        ax.plot(MT_R * np.cos(th), MT_R * np.sin(th), "--", color="#e34948", lw=1.2)
        ax.plot(ST.x[mi.PERMANENT], ST.y[mi.PERMANENT], "^", color="#8a8984", ms=3)
        if g in TILED:
            ax.plot(ST.x[f["node_ids"]], ST.y[f["node_ids"]], ".", color="k", ms=1.5)
        ax.set(aspect="equal", xlim=(mi.IX[0], mi.IX[-1]), ylim=(mi.IY[0], mi.IY[-1]),
               title=f"{d}\narea coverage {sumC.area_coverage_mean[d]:.2f}")
        ax.title.set_fontsize(8)
    for ax in axs.ravel()[len(designs_C):]:
        ax.axis("off")
    fig.colorbar(pc, ax=axs, label="fraction of conductor positions imaged correctly", shrink=0.7)
    fig.suptitle(f"Red dashed: MT target area ({MT_R:g} km). Grey: Trillium/Mercury. Black: tiled nodes", fontsize=9)
    plt.show()

In [ ]:
# --- One conductor position in detail: picked minus true depth. White = right, red = too shallow, blue = too deep,
#     grey = not imaged ---
SHOW_C = SHOW_SCENARIO
with plt.style.context("default"):
    fig, axs = plt.subplots(nr, nc, figsize=(3.6 * nc + 0.8, 3.5 * nr), sharex=True, sharey=True,
                            squeeze=False, layout="constrained")
    for ax, d in zip(axs.ravel(), designs_C):
        g, K, a, b = key_of[d]
        f = load(SHOW_C, 0, g, K, a, b)
        err = mi.pick_top(f["final"]) - mi.true_top(SHOW_C)
        ax.set_facecolor("#d9d8d3")
        pc = ax.pcolormesh(mi.IX, mi.IY, err.T, cmap="RdBu", vmin=-1, vmax=1, shading="auto")
        ax.contour(mi.IX, mi.IY, mi.true_top(SHOW_C).T, levels=np.arange(4, 7, 0.5), colors="#52514e",
                   linewidths=0.5)
        ax.plot(MT_R * np.cos(th), MT_R * np.sin(th), "--", color="#0b0b0b", lw=1)
        st = f["node_ids"]
        ax.plot(ST.x[st], ST.y[st], ".", color="k", ms=1.5)
        ax.plot(*mi.SC["scenarios"][SHOW_C]["centre"], "+", color="#0b0b0b", ms=10, mew=2)
        row = resC[(resC.design == d) & (resC.scenario == SHOW_C) & (resC.real == 0)].iloc[0]
        ax.set(aspect="equal", xlim=(mi.IX[0], mi.IX[-1]), ylim=(mi.IY[0], mi.IY[-1]),
               title=f"{d}\narea coverage {row.area_coverage:.2f}")
        ax.title.set_fontsize(8)
    for ax in axs.ravel()[len(designs_C):]:
        ax.axis("off")
    fig.colorbar(pc, ax=axs, label="picked - true BDT top (km)", shrink=0.7)
    fig.suptitle(f"{SHOW_C}: + = BDT high, grey lines = true top every 0.5 km, dashed = MT area", fontsize=9)
    plt.show()

## Reading the results

- **Moves:** compare each K with the dashed 6-move line in test A. If 3 moves are within the noise-realisation
  spread of 6, the extra trips are not buying imaging.
- **Layout:** `interleaved` puts long and short offsets in every deployment, `rolling` gives dense contiguous
  coverage but each deployment sees a small aperture, and `zoom` spends one move to fix a poor phase 1 pick.
  `zoom` should win when the phase 1 error (`p1_error_km`) is large, i.e. short phase 1 or high noise.
- **Whole area (test C):** the phased designs image the crest well but are judged here on the whole 4 km
  circle. If a tiled design has the higher `area_coverage` *and* a reasonable crest score, it is the safer
  choice for 3 trips. The reliability map shows where each design fails: look for blue covering the dashed circle.
- **Length:** at fixed trips, longer deployments only help while noise limits the image. Look for where the
  test B heatmap flattens. Compare schedules of the same campaign length (e.g. 2 x 56 d against 4 x 28 d) in the
  last plot to see whether a fourth trip beats longer recording.

Caveats: the noise level (`NSR`) is unknown (06 scans it), so rerun with `NSR = 0.5` and `2.0` before choosing.
`NODE_BATTERY_DAYS` should be set to the nodes actually used. The defaults are sized to run quickly: two dome
positions and one noise realisation. Before settling on a schedule, add `pos02_dome` and set `N_REAL = 2-3`, and
consider `FAST = None` for 05's grids.